In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage 

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-2.5-flash",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [32]:
config = {"configurable":{"thread_id":"test-3"}}

questions = [
    "What is 2+2?",
    "What is 5+3?",
    "What is 10-4?",
    "What is 6*2?",
    "What is 15/3?",
    "What is 9+7?"
]


In [33]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(
        content=q
    )]},config)

    print(f"message :{response["messages"][-1].content}")
    print(f"Total message :{len(response["messages"])}")

message :2 + 2 = 4
Total message :2
message :5 + 3 = 8
Total message :4
message :10 - 4 = 6
Total message :6
message :6 * 2 = 12
Total message :8
message :15 / 3 = 5
Total message :10
message :9 + 7 = 16
Total message :6


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotel(city:str)->str:
    """search hotels-returns lang response to use more tokens"""
    return f"""Hotels in {city}: 
            1. grand hotel -5 star , $350/night , spa , pool , gym 
            2. city inn -4 star , $180/night , bussiness center 
            3. budget stay -3 star , $75/night , free wifi  
    """

agent = create_agent(
     model="google_genai:gemini-2.5-flash",
     tools=[search_hotel] ,\
     checkpointer=InMemorySaver(),
     middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash",
        trigger=("tokens",550),
        keep=("tokens",200)
     )]

)

In [ ]:
config = {"configurable":{"tread-id":"test-1"}}

def count_token(messages):
    total_chars = sum(len(str(m.context)) for m in messages)
    return total_chars
    